# Machine Translation Demo with LLMs (EN → FR)


## Setup

Requires [Ollama](https://ollama.com/download) running locally with `llama3` pulled:

```bash
ollama pull llama3
```

In [3]:
# Install dependencies (uncomment if needed)
%pip install datasets transformers torch sentencepiece sacrebleu rouge-score langchain-ollama langchain-core pandas --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import sacrebleu
from rouge_score import rouge_scorer
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

pd.set_option("display.max_colwidth", 80)

/Users/Licas/Desktop/github/NLP/nlpenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Benchmark translation dataset

We use [`opus_books`](https://huggingface.co/datasets/Helsinki-NLP/opus_books) (English–French), a standard parallel corpus where each row contains an aligned sentence pair.
We keep a small subset for speed: `original` = English source, `reference` = human French translation (our ground truth).

In [5]:
# Load a subset of the EN-FR benchmark
dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

# Build (original, reference) pairs, keeping reasonably short sentences for a readable demo
pairs = []
for ex in dataset:
    en = ex["translation"]["en"].strip()
    fr = ex["translation"]["fr"].strip()
    if 30 <= len(en) <= 120 and 30 <= len(fr) <= 140:
        pairs.append({"original": en, "reference": fr})
    if len(pairs) >= 10:
        break

df_data = pd.DataFrame(pairs)
print(f"{len(df_data)} sentence pairs loaded")
df_data

10 sentence pairs loaded


,original,reference
0,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…
1,"I still say 'our home,' although the house no longer belongs to us.","Je continue à dire « chez nous », bien que la maison ne nous appartienne plus."
2,We left that part of the country nearly fifteen years ago and shall certainl...,Nous avons quitté le pays depuis bientôt quinze ans et nous n’y reviendrons ...
3,We were living in the building of the Higher Elementary Classes at Sainte-Ag...,Nous habitions les bâtiments du Cours Supérieur de Sainte-Agathe.
4,"At the time of some new 'appointments,' a whim of fate, due to some inspecto...","Le hasard des « changements », une décision d’inspecteur ou de préfet nous a..."
5,Thus to-day I picture our arrival.,"C’est ainsi, du moins, que j’imagine aujourd’hui notre arrivée."
6,Yet we had already been ten years in that district when Meaulnes arrived.,Nous étions pourtant depuis dix ans dans ce pays lorsque Meaulnes arriva.
7,"It was a cold Sunday of November, the first day of autumn to make one think ...","C’était un froid dimanche de novembre, le premier jour d’automne qui fît son..."
8,All day Millie had waited for the station omnibus to bring her a hat for the...,"Toute la journée, Millie avait attendu une voiture de La Gare qui devait lui..."
9,"In the afternoon, I had to go to vespers alone.","Après midi, je dus partir seul à vêpres."


## Step 2 — Translators: one Ollama LLM + one HuggingFace LLM

* **Ollama** → `llama3` (general instruction-tuned LLM, prompted to translate).
* **HuggingFace** → `Helsinki-NLP/opus-mt-en-fr` (MarianMT, a dedicated EN→FR translation model).

Both expose a `temperature` argument so we can sweep it in Step 3. For `temperature = 0` we use greedy decoding (deterministic).

In [6]:
# --- Ollama translator (llama3) ---
def translate_ollama(sentence, temperature, model_name="llama3"):
    llm = ChatOllama(model=model_name, temperature=temperature)
    prompt = (
        "Translate the following English sentence into French.\n"
        "Provide only the French translation, with no explanations, notes or quotes.\n\n"
        f"{sentence}"
    )
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()


# --- HuggingFace translator (MarianMT EN->FR) ---
HF_NAME = "Helsinki-NLP/opus-mt-en-fr"
hf_tokenizer = MarianTokenizer.from_pretrained(HF_NAME)
hf_model = MarianMTModel.from_pretrained(HF_NAME)


def translate_hf(sentence, temperature):
    inputs = hf_tokenizer(sentence, return_tensors="pt", truncation=True)
    gen_kwargs = {"max_length": 128, "num_beams": 1}
    if temperature and temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=float(temperature))
    else:
        gen_kwargs.update(do_sample=False)  # greedy = deterministic
    output = hf_model.generate(**inputs, **gen_kwargs)
    return hf_tokenizer.decode(output[0], skip_special_tokens=True).strip()

/Users/Licas/Desktop/github/NLP/nlpenv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [7]:
# Quick sanity check on the first sentence
sample = df_data.iloc[0]["original"]
print("EN:        ", sample)
print("Reference: ", df_data.iloc[0]["reference"])
print("Ollama:    ", translate_ollama(sample, temperature=0.2))
print("HuggingFace:", translate_hf(sample, temperature=0.2))

EN:         He arrived at our home on a Sunday of November, 189-.
Reference:  Il arriva chez nous un dimanche de novembre 189-…
Ollama:     Il arriva à notre maison le dimanche du mois de novembre 189-.
HuggingFace: Il est arrivé chez nous un dimanche de novembre 189.


## Step 3 — Generate translations across temperature settings

We run every sentence through both engines at each temperature in `{0, 0.2, 0.5, 0.8, 1.0}`.

In [8]:
temperatures = [0, 0.2, 0.5, 0.8, 1.0]
engines = {
    "ollama:llama3": translate_ollama,
    "hf:opus-mt-en-fr": translate_hf,
}

results = []
for i, row in df_data.iterrows():
    for temp in temperatures:
        for engine_name, translate_fn in engines.items():
            try:
                translation = translate_fn(row["original"], temp)
            except Exception as e:
                print(f"Error {engine_name} @ {temp}: {e}")
                translation = ""
            results.append({
                "id": i,
                "engine": engine_name,
                "temperature": temp,
                "original": row["original"],
                "reference": row["reference"],
                "translation": translation,
            })
    print(f"Sentence {i + 1}/{len(df_data)} done")

df_results = pd.DataFrame(results)
print(f"\n{len(df_results)} translations generated")
df_results.head()

Sentence 1/10 done
Sentence 2/10 done
Sentence 3/10 done
Sentence 4/10 done
Sentence 5/10 done
Sentence 6/10 done
Sentence 7/10 done
Sentence 8/10 done
Sentence 9/10 done
Sentence 10/10 done

100 translations generated


,id,engine,temperature,original,reference,translation
0,0,ollama:llama3,0.0,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il arriva à notre maison le dimanche de novembre 189-.
1,0,hf:opus-mt-en-fr,0.0,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il est arrivé à notre maison un dimanche de novembre 189-.
2,0,ollama:llama3,0.2,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il arriva à notre maison le dimanche de novembre 189-.
3,0,hf:opus-mt-en-fr,0.2,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il est arrivé chez nous un dimanche de novembre 189.
4,0,ollama:llama3,0.5,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il est arrivé à notre maison le dimanche du mois de novembre 189-.


## Step 4 — Evaluation with BLEU and ROUGE

Each generated translation is compared to the human French reference.
* **BLEU** (sacreBLEU) — n-gram precision, the standard MT metric.
* **ROUGE-1 / ROUGE-L** — unigram and longest-common-subsequence overlap (recall-oriented).

In [9]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)


def evaluate(reference, hypothesis):
    if not hypothesis:
        return 0.0, 0.0, 0.0
    bleu = sacrebleu.sentence_bleu(hypothesis, [reference]).score
    scores = scorer.score(reference, hypothesis)
    return bleu, scores["rouge1"].fmeasure, scores["rougeL"].fmeasure


df_results[["BLEU", "ROUGE-1", "ROUGE-L"]] = df_results.apply(
    lambda r: pd.Series(evaluate(r["reference"], r["translation"])), axis=1
)
df_results.head()

,id,engine,temperature,original,reference,translation,BLEU,ROUGE-1,ROUGE-L
0,0,ollama:llama3,0.0,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il arriva à notre maison le dimanche de novembre 189-.,36.462859,0.666667,0.666667
1,0,hf:opus-mt-en-fr,0.0,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il est arrivé à notre maison un dimanche de novembre 189-.,39.553325,0.631579,0.631579
2,0,ollama:llama3,0.2,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il arriva à notre maison le dimanche de novembre 189-.,36.462859,0.666667,0.666667
3,0,hf:opus-mt-en-fr,0.2,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il est arrivé chez nous un dimanche de novembre 189.,59.004687,0.842105,0.842105
4,0,ollama:llama3,0.5,"He arrived at our home on a Sunday of November, 189-.",Il arriva chez nous un dimanche de novembre 189-…,Il est arrivé à notre maison le dimanche du mois de novembre 189-.,18.207053,0.476190,0.476190


In [10]:
# Summary: mean scores per engine and temperature
summary = (
    df_results
    .groupby(["engine", "temperature"])
    .agg({"BLEU": "mean", "ROUGE-1": "mean", "ROUGE-L": "mean"})
    .round(2)
)
summary

BLEU  ROUGE-1  ROUGE-L
engine           temperature                         
hf:opus-mt-en-fr 0.0          23.34     0.59     0.57
                 0.2          24.53     0.60     0.58
                 0.5          23.92     0.58     0.56
                 0.8          17.05     0.55     0.52
                 1.0          16.56     0.55     0.52
ollama:llama3    0.0          23.17     0.58     0.53
                 0.2          23.23     0.56     0.52
                 0.5          22.24     0.56     0.51
                 0.8          19.90     0.55     0.51
                 1.0          14.97     0.54     0.49

In [11]:
# Best configuration by BLEU
best = summary.sort_values("BLEU", ascending=False)
print("Ranking by mean BLEU:\n")
print(best)

# Save raw results for the report
df_results.to_csv("translation_results.csv", index=False)
print("\nSaved -> translation_results.csv")

Ranking by mean BLEU:

                               BLEU  ROUGE-1  ROUGE-L
engine           temperature                         
hf:opus-mt-en-fr 0.2          24.53     0.60     0.58
                 0.5          23.92     0.58     0.56
                 0.0          23.34     0.59     0.57
ollama:llama3    0.2          23.23     0.56     0.52
                 0.0          23.17     0.58     0.53
                 0.5          22.24     0.56     0.51
                 0.8          19.90     0.55     0.51
hf:opus-mt-en-fr 0.8          17.05     0.55     0.52
                 1.0          16.56     0.55     0.52
ollama:llama3    1.0          14.97     0.54     0.49

Saved -> translation_results.csv


In [12]:
# A few qualitative examples
for engine_name in engines:
    print(f"\n=== {engine_name} ===")
    sub = df_results[(df_results.engine == engine_name) & (df_results.temperature == 0.2)].head(3)
    for _, row in sub.iterrows():
        print(f"EN:    {row['original']}")
        print(f"REF:   {row['reference']}")
        print(f"MT:    {row['translation']}")
        print(f"BLEU: {row['BLEU']:.2f} | ROUGE-1: {row['ROUGE-1']:.2f} | ROUGE-L: {row['ROUGE-L']:.2f}\n")


=== ollama:llama3 ===
EN:    He arrived at our home on a Sunday of November, 189-.
REF:   Il arriva chez nous un dimanche de novembre 189-…
MT:    Il arriva à notre maison le dimanche de novembre 189-.
BLEU: 36.46 | ROUGE-1: 0.67 | ROUGE-L: 0.67

EN:    I still say 'our home,' although the house no longer belongs to us.
REF:   Je continue à dire « chez nous », bien que la maison ne nous appartienne plus.
MT:    Je dis toujours « notre maison », bien que la maison ne nous appartienne plus.
BLEU: 59.99 | ROUGE-1: 0.69 | ROUGE-L: 0.69

EN:    We left that part of the country nearly fifteen years ago and shall certainly never go back to it.
REF:   Nous avons quitté le pays depuis bientôt quinze ans et nous n’y reviendrons certainement jamais.
MT:    Nous avons quitté cette partie du pays il y a près de quinze ans et ne retournerons jamais à elle.
BLEU: 12.58 | ROUGE-1: 0.49 | ROUGE-L: 0.43


=== hf:opus-mt-en-fr ===
EN:    He arrived at our home on a Sunday of November, 189-.
REF:   Il ar